In [7]:
!pip install pandas matplotlib seaborn scrapy


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%%writefile spider.py
import scrapy
from scrapy.crawler import CrawlerProcess


class BooksSpider(scrapy.Spider):
    name = "books_spider"
    start_urls = ["https://books.toscrape.com/catalogue/page-1.html"]
    page_count = 1

    def parse(self, response):
        book_links = response.css("article.product_pod h3 a::attr(href)").getall()
        for link in book_links:
            yield response.follow(link, callback=self.parse_book_page)

        next_page = response.css("li.next a::attr(href)").get()
        if next_page and self.page_count < 5:
            self.page_count += 1
            yield response.follow(next_page, callback=self.parse)

    def parse_book_page(self, response):
        table_data = {}
        for row in response.css("table.table-striped tr"):
            key = row.css("th::text").get()
            val = row.css("td::text").get()
            if key and val:
                table_data[key.strip()] = val.strip()

        item = {
            "title": response.css("div.product_main h1::text").get(),
            "category": response.css("ul.breadcrumb li:nth-child(3) a::text").get(),
            "price": response.css("p.price_color::text").get(),
            "rating": response.css("p.star-rating::attr(class)").get().replace("star-rating ", "") if response.css("p.star-rating") else None,
            "availability": response.css("p.instock.availability::text").getall(),
            "product_description": response.css("#product_description + p::text").get(),
            "upc": table_data.get("UPC"),
            "number_of_reviews": table_data.get("Number of reviews"),
            "product_url": response.url
        }
        
        if isinstance(item["availability"], list):
            item["availability"] = " ".join([text.strip() for text in item["availability"] if text.strip()])

        yield item

Writing spider.py


In [3]:
import sys
import subprocess

# Run Scrapy through the active Python interpreter
result = subprocess.run([sys.executable, "-m", "scrapy", "runspider", "spider.py", "-o", "books_raw.csv", "-t", "csv"], capture_output=True, text=True)

print("--- Output ---")
print(result.stdout[-500:] if result.stdout else "Scraping finished with no errors.")

print("\n--- Errors/Logs (Last 10 lines) ---")
print("\n".join(result.stderr.splitlines()[-10:]))

--- Output ---
       log file. if omitted stderr will be used
  -L, --loglevel LEVEL  log level (default: DEBUG)
  --nolog               disable logging completely
  --profile FILE        write python cProfile stats to FILE
  --pidfile FILE        write process ID to FILE
  -s, --set NAME=VALUE  set/override setting (may be repeated)
  --pdb                 enable pdb on failure


--- Errors/Logs (Last 10 lines) ---
 'parsel': '1.11.0',
 'w3lib': '2.4.1',
 'Twisted': '26.4.0',
 'Python': '3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 '
           '64 bit (AMD64)]',
 'pyOpenSSL': '26.4.0 (OpenSSL 4.0.1 9 Jun 2026)',
 'cryptography': '50.0.0',
 'Platform': 'Windows-11-10.0.26200-SP0'}
2026-08-06 19:54:35 [scrapy.crawler] DEBUG: Using AsyncCrawlerProcess
2026-08-06 19:54:35 [asyncio] DEBUG: Using selector: SelectSelector


In [4]:
import sys
import subprocess

# Run Scrapy using the active Python interpreter
result = subprocess.run([sys.executable, "-m", "scrapy", "runspider", "spider.py", "-o", "books_raw.csv", "-t", "csv"], capture_output=True, text=True)

print("Scraping finished!")

Scraping finished!
